# Example 9.2 / 1: Read Keys into a Set and Rename Columns

Companion notebook for **9.2 Reading Tables** in the AMPL lab Documentation, section "Read Keys into a Set and Rename Columns". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read Keys into a Set and Rename Columns

Use the left arrow when the table's keys define set membership, and the correspondence operator
`~` when an AMPL name and an external heading differ. Here the file uses business headings
`unit_profit` and `max_sales`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products, filled from the product column
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # maximum sales, tons

table Catalog IN "amplcsv" "catalog.csv":
    PROD <- [product],
    profit ~ unit_profit,
    market ~ max_sales;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("catalog.csv", "w") as f:
    f.write("product,unit_profit,max_sales\n"
            "bands,25,6000\n"
            "coils,30,4000\n"
            "plate,29,3500\n")

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.read_table("Catalog")
print(ampl.set["PROD"].to_list())
print(ampl.param["profit"].to_dict())

['bands', 'coils', 'plate']
{'bands': 25, 'coils': 30, 'plate': 29}


### Pure AMPL: model.run

The read establishes `PROD = {bands, coils, plate}` and assigns both indexed parameters. In each
`~` pair the AMPL expression is on the left and the external heading on the right, so model
names stay stable even when a business-facing schema uses different headings.

Pure AMPL, with the same `model.mod` and `catalog.csv`:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
with open("catalog.csv", "w") as f:
    f.write("product,unit_profit,max_sales\n"
            "bands,25,6000\n"
            "coils,30,4000\n"
            "plate,29,3500\n")

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
read table Catalog;
display PROD, profit;

set PROD := bands coils plate;

profit [*] :=
bands  25
coils  30
plate  29
;

